In [1]:
# 6c_nan_report.ipynb
#
# Reports NaN and negative-value counts per column in the synthetic population parquet.
# Also cross-checks which config-defined feature columns are present / missing.

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_paths as _cp
importlib.reload(_cp)
from data_pipeline.config_paths import DATA_FOLDER
import data_pipeline.config_variables as _cv
_cv.reload_config_variables()
from data_pipeline.config_variables import VARIABLES
from data_pipeline.config_cluster import WAVE

import pandas as pd
import numpy  as np
from pathlib import Path

# ── Load ──────────────────────────────────────────────────────────────────────
SYNPOP_PARQUET = Path(f"../{DATA_FOLDER}/6_synthetic_population/synthetic_population.parquet")
if not SYNPOP_PARQUET.exists():
    raise FileNotFoundError(f"{SYNPOP_PARQUET} — run 6a_synthetic_population.ipynb first.")

print(f"Loading {SYNPOP_PARQUET} ...")
df = pd.read_parquet(SYNPOP_PARQUET)
n_rows, n_cols = len(df), len(df.columns)
print(f"  {n_rows:,} rows x {n_cols} columns")

# ── Config vs parquet cross-check ─────────────────────────────────────────────
expected = {f"{WAVE}_{b}" for b in VARIABLES}
missing  = sorted(expected - set(df.columns))
extra    = sorted(set(df.columns) - expected)

print(f"\nConfig feature columns ({WAVE}_*): {len(expected)}")
print(f"  Present : {len(expected) - len(missing)}")
if missing:
    print(f"  Missing : {len(missing)}")
    for c in missing:
        print(f"    x {c}")
print(f"  Extra (geography / OHE etc.): {len(extra)}")
for c in extra:
    print(f"    + {c}")

# ── NaN report ────────────────────────────────────────────────────────────────
print("\n-- NaN report --")
nan_counts = df.isna().sum()
nan_cols   = nan_counts[nan_counts > 0].sort_values(ascending=False)
if nan_cols.empty:
    print("No NaNs in any column.")
else:
    print(f"{len(nan_cols)} column(s) with NaNs:")
    display(pd.DataFrame({
        "nan_count":   nan_cols,
        "nan_percent": (nan_cols / n_rows * 100).round(2),
    }).rename_axis("column"))

# ── Negative-value report ─────────────────────────────────────────────────────
print("\n-- Negative-value report --")
num_cols   = df.select_dtypes(include=[np.number]).columns
neg_counts = df[num_cols].apply(lambda s: (s < 0).sum())
neg_cols   = neg_counts[neg_counts > 0].sort_values(ascending=False)
if neg_cols.empty:
    print(f"No negative values in any of the {len(num_cols)} numeric columns.")
else:
    print(f"{len(neg_cols)} column(s) with negative values:")
    display(pd.DataFrame({
        "neg_count":   neg_cols,
        "neg_percent": (neg_cols / n_rows * 100).round(2),
        "min_value":   df[neg_cols.index].min().round(4),
    }).rename_axis("column"))



🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  
  FOUR-LA SUBSET ACTIVE — Newham, Tower Hamlets, Islington, Hounslow only (4 LAs)
  Set USE_FOUR_LA_SUBSET = False for all London or full UK.
🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  

Loading ../data/6_synthetic_population/synthetic_population.parquet ...
  525,150 rows x 103 columns

Config feature columns (o_*): 9
  Present : 8
  Missing : 1
    x o_tenure_dv
  Extra (geography / OHE etc.): 95
    + ladcd
    + ladnm
    + msoa21cd
    + msoa21nm
    + o_benbase4
    + o_browse
    + o_caruse
    + o_derived_work_status
    + o_digital_use
    + o_drive
    + o_email
    + o_envhabit8
    + o_fimngrs_dv
    + o_fimnsben_dv
    + o_hhtype_dv_0
    + o_hhtype_dv_1
    + o_hhtype_dv_10
    + o_hhtype_dv_11
    + o_hhtype_dv_12
    + o_hhtype_dv_13
    + o_hhtype_dv_14
    + o_hhtype_dv_2
    + o_hhtype_dv_3
    + o_hhtype_dv_4
    + o_hhtype_dv_5
    + o_hhtype_dv_6


,nan_count,nan_percent
column,,
o_derived_work_status,183590,34.96
o_locsere,68714,13.08
o_locserc,61342,11.68
o_locserd,61041,11.62
o_nbrsnci_dv,43512,8.29
o_email,11966,2.28
o_browse,11958,2.28
o_onlinebank,11958,2.28
o_onlinebuy,11958,2.28



-- Negative-value report --
26 column(s) with negative values:


,neg_count,neg_percent,min_value
column,,,
o_jbttwt,317949,60.54,-8.000000
o_payn_dv,296585,56.48,-9.000000
o_servuse1,292949,55.78,-9.000000
o_servuse2,292949,55.78,-9.000000
o_servuse10,292949,55.78,-9.000000
o_jbpl,257005,48.94,-9.000000
o_jbnssec8_dv,241090,45.91,-9.000000
o_drive,132816,25.29,-8.000000
o_derived_work_status,67304,12.82,-9.000000
